# Scenario 1: <sup>1</sup>H-NMR kinetic analysis of Pd(0)-catalyzed cycloisomerization

## Data Modeling Notebook

---

The educt concentration time courses from the analysis notebook (`cycloisomerization_analysis.ipynb`) are fitted with a kinetic model by Bayesian inference, using Catalax. The fitted model and its parameters are saved as their own JSON file and written into an EnzymeML document.

### Notebook Setup

The essential packages to work with the modeling Notebook are:

- `catalax`
- `pyenzyme`

The following packages in this Notebook are used for convenience:

- `math`
- `pathlib`
- `rich`

Depending on the use case, some additional packages may be useful:

- `numpy`
- `numpyro`
- `jax.numpy`
- `...`

The Markov chains of the Hamiltonian Monte Carlo (HMC) sampler below run in parallel, one per CPU core made available to `numpyro` here.

In [ ]:
import numpyro
numpyro.set_host_device_count(8)

In [ ]:
from math import e
from pathlib import Path
from rich import print

import catalax as ctx
import catalax.mcmc as mcmc
import jax.numpy as jnp
import pyenzyme as pe

---

### Setting up the modeling environment

Besides loading the EnzymeML document from the analysis notebook, the modeling environment has to be set up. This includes defining the model, the parameters to be estimated, and the data to be used for fitting.

#### EnzymeML document

`SOURCE` selects the document that is fitted: `"output"` (default) is the result shipped in `data/output/`, `"run"` is the document the analysis notebook writes to `data/run/`. Both hold all three measurements (2, 5 and 33 eq BINOL) with their educt concentration time courses, corrected for the baseline floor as described in the analysis notebook. The results of this notebook are written to `data/run/`.

In [ ]:
SOURCE = "output"  # "output": result shipped in data/output/, "run": result of the analysis notebook

path_to_enzymeml = Path.cwd() / "data" / SOURCE / "cycloisomerization.json"

RUN_DIR = Path.cwd() / "data/run"
RUN_DIR.mkdir(exist_ok=True)
path_to_modeled_enzymeml = RUN_DIR / "cycloisomerization_modeled.json"

In [ ]:
doc = pe.read_enzymeml(path_to_enzymeml)

The species not relevant to the modeling are removed, so that they do not interfere with the fitting process and do not show up in the plots: the product (s2), the internal standard mesitylene (s5), and s6. What remains is the educt (s1), Pd(0) (s3) and BINOL (s4).

In [ ]:
SPECIES_TO_REMOVE = ["s2", "s5", "s6"]

doc.small_molecules = [s for s in doc.small_molecules if s.id not in SPECIES_TO_REMOVE]

for m in doc.measurements:
    m.species_data = [d for d in m.species_data if d.species_id not in SPECIES_TO_REMOVE]

In [ ]:
pe.summary(doc)

#### Catalax dataset

Due to the way Catalax works, species that enter another species' rate equation but do not receive a rate equation themselves have to be removed from the dataset's states. Their prepared concentrations still enter the model as constants (see below).

Here, both Pd(0) (s3) and BINOL (s4) are removed from the dataset's states and from each measurement's states and data.

In [ ]:
dataset = ctx.Dataset.from_enzymeml(doc)
dataset = dataset.pad()

In [ ]:
dataset.states.pop()
dataset.states.pop()
for m in dataset.measurements:
    m.data.popitem()
    m.data.popitem()
    m.state.pop()
    m.state.pop()

In [ ]:
dataset.plot(show=True)

#### Catalax model

The model describes the consumption of the educt (s1) as a pseudo-first-order, irreversible reaction catalyzed by Pd(0) (s3) and BINOL (s4), plus a background reaction:

$$\frac{d[s1]}{dt} = -\left(\frac{k_{cyc} \cdot K_{eq} \cdot [s3] \cdot [s4] \cdot [s1]}{1 + K_{eq} \cdot [s4]} + k_{bkgrnd} \cdot [s1]\right)$$

The equilibrium constant $K_{eq}$ at the reaction temperature of 80 °C is calculated from the thermodynamic parameters. Uniform priors are set for the two parameters to be estimated, $k_{cyc}$ and $k_{bkgrnd}$.

In [ ]:
model = ctx.Model(name="Pseudo 1st order irreversible with background")
model.add_state("s1")
model.add_constants(s3="s3", s4="s4")

R = 1.987    # cal/(mol*K)
dH = -10200  # cal/mol
dS = -30     # cal/(mol*K)
T = 353.15   # K
KEQ_80 = e**( -(dH/R)*(1/T)+(dS/R) )  # ln(KEQ) = -(dH/R)*(1/T) + dS/R

model.add_ode(
    "s1",
    "-((( k_cyc * KEQ * s3 * s4 * s1 ) / "
    "( 1 + KEQ * s4 )) + k_bkgrnd * s1 )".replace("KEQ", str(KEQ_80)),
)

model.parameters["k_cyc"].prior = mcmc.priors.Uniform(low=1, high=5e3)
model.parameters["k_bkgrnd"].prior = mcmc.priors.Uniform(low=1e-2, high=0.5)

model

#### HMC run

Finally, the HMC sampler is run to estimate the parameters of the model. With 8 chains of 10,000 warm-up and 10,000 sampling steps each, this takes a while.

In [ ]:
hmc = mcmc.HMC(
    num_warmup=10000,
    num_samples=10000,
    dt0=0.1,
    max_steps=64**4,
    verbose=1,
    num_chains=8,
    chain_method="parallel",
)

results = hmc.run(model=model, dataset=dataset, yerrs=2.0)

# Apply the median posterior to the model
fitted_model = results.get_fitted_model()

# Print the summary
results.summary()

---

### Results

Catalax provides various plotting functions to visualize the results of the HMC run. Here, the median of the posterior samples is printed, followed by the effective sample size (ESS), the corner plot, the trace plot, and the fitted model.

In [ ]:
print("Median of the posterior samples")
print(f"k_cyc:    {jnp.median(results.get_samples()['k_cyc'])}")
print(f"k_bkgrnd: {jnp.median(results.get_samples()['k_bkgrnd'])}")

In [ ]:
results.plot_ess(show=True)

In [ ]:
results.plot_corner(show=True)

In [ ]:
results.plot_trace(show=True)

In [ ]:
dataset.plot(
    predictor=fitted_model,
    measurement_ids=[m.id for m in dataset.measurements[:3]],
    show=True,
)

#### Save fitted model

The model can be saved as its own data model in JSON format, including the uncertainty of its parameters.

In [ ]:
fitted_model.save(str(RUN_DIR), "fitted_model")

#### Update EnzymeML document

The EnzymeML document can be updated with the fitted model and the estimated parameters. This keeps all information about the experiment in one place and makes it easy to share with others. The updated document is written to `data/run/`, next to the fitted model, so that the document it was read from stays unchanged and this notebook can be run again.

In [ ]:
updated_doc = fitted_model.to_enzymeml(doc)
pe.summary(updated_doc)

In [ ]:
pe.write_enzymeml(updated_doc, path_to_modeled_enzymeml)

---

### Disclosure

**Contributions**

If you wish to contribute to the EnzymeML and/or NMRPy platforms, find us on our [EnzymeML GitHub](https://github.com/EnzymeML) and [NMRPy GitHub](https://github.com/NMRPy)!

**BSD 3-Clause License**

Copyright (c) 2026 Torsten Giess

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its contributors may be used to endorse or promote products derived from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS “AS IS” AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.